In [ ]:
import sys                      # Permet de configurer les chemins des imports
from pathlib import Path        # Permet de manipuler les chemins de fichiers

project_root = Path.cwd()       # Récupère le dossier de travail de Jupyter

if not (project_root / "scripts").is_dir():  # Si on n'est pas déjà à la racine
    project_root = project_root.parent.parent  # Remonte depuis notebooks/fine_tuning

assert (project_root / "scripts").is_dir(), "Vérifie le dossier de travail."

if str(project_root) not in sys.path:       # Évite d'ajouter deux fois le chemin
    sys.path.insert(0, str(project_root))   # Rend les scripts du projet importables

print("Racine du projet :", project_root)  # Affiche le chemin trouvé

In [ ]:
import torch                           # Bibliothèque utilisée pour entraîner le modèle
from transformers import set_seed      # Fonction qui fixe les graines aléatoires

SEED = 42                              # Même valeur pour reproduire nos expériences
set_seed(SEED)                         # Fixe le hasard de Python, NumPy et PyTorch

cuda_available = torch.cuda.is_available()  # Vérifie si PyTorch peut utiliser CUDA

print("Version PyTorch :", torch.__version__)  # Affiche la version installée
print("CUDA disponible :", cuda_available)     # Affiche True ou False

if cuda_available:                            # Si un GPU CUDA est accessible
    print("GPU :", torch.cuda.get_device_name(0))  # Affiche le nom du premier GPU

In [ ]:
# Chemin des annotations dans l'environnement d'origine
DATA_DIR = Path("/mnt/imported/data/sav-label-studio/Tasks-intention-transaction/OutputTasks")

assert DATA_DIR.is_dir(), f"Dossier introuvable : {DATA_DIR}"  # Vérifie le chemin

data_files = sorted(
    path for path in DATA_DIR.rglob("*") if path.is_file()
)  # Liste les fichiers du dossier et de ses sous-dossiers

print("Nombre de fichiers :", len(data_files))  # Compte les fichiers trouvés

for path in data_files[:5]:   # Prend les cinq premiers fichiers
    print(path.name)          # Affiche leur nom

In [ ]:
import json  # Permet de lire les fichiers JSON

assert data_files, "Aucun fichier trouvé."  # Vérifie que la liste n'est pas vide

example_path = data_files[0]  # Choisit le premier fichier de la liste

with example_path.open("r", encoding="utf-8") as file:  # Ouvre le fichier en lecture
    raw_example = json.load(file)                    # Charge son contenu en Python

print(
    json.dumps(raw_example, indent=2, ensure_ascii=False)
)  # Affiche le contenu lisiblement, en conservant les accents

In [ ]:
def get_choices(example, field_name):          # Recherche un champ d'annotation
    for annotation in example["result"]:      # Parcourt les annotations du message
        if annotation["from_name"] == field_name:  # Repère le champ demandé
            return annotation["value"].get("choices", [])  # Renvoie ses choix

    return []  # Renvoie une liste vide si le champ n'existe pas


text = raw_example["task"]["data"]["message"]  # Récupère le texte du message
intents = get_choices(raw_example, "intents")  # Récupère toutes ses intentions
intent_types = get_choices(raw_example, "type_intent")  # Récupère son type

print("Texte :", text)                        # Affiche le message
print("Intentions :", intents)                # Affiche la liste des intentions
print("Types d'intention :", intent_types)    # Affiche les choix du champ type

In [ ]:
def prepare_example(example):  # Transforme une annotation en un exemple simplifié
    if example["was_cancelled"]:  # Vérifie si l'annotation a été annulée
        return None               # Indique qu'on ne garde pas cet exemple

    review = get_choices(example, "review_decision")  # Récupère la décision de relecture

    if review != ["Accepter"]:  # Écarte les annotations non acceptées
        return None

    data = example["task"]["data"]  # Récupère les données du message
    intents = get_choices(example, "intents")  # Conserve toutes les intentions
    intent_types = get_choices(example, "type_intent")  # Récupère les types sélectionnés

    if not intents:  # Vérifie qu'au moins une intention est renseignée
        raise ValueError("Cet exemple accepté n'a aucune intention.")

    if len(intent_types) != 1:  # Vérifie qu'il y a exactement un type d'intention
        raise ValueError(f"Un seul type attendu, trouvé : {intent_types}")

    assimilated = get_choices(example, "Assimilated_by_Mode")  # Garde cette information pour le filtrage

    return {  # Construit le dictionnaire représentant notre exemple
        "text": data["message"],              # Texte donné au modèle
        "intents": intents,                   # Liste des intentions à prédire
        "type_intent": intent_types[0],       # Unique type à prédire
        "split": data["split"],               # Groupe d'origine : train ou test
        "Assimilated_by_Mode": assimilated[0] if assimilated else None,  # None si absent
    }


prepared_example = prepare_example(raw_example)  # Applique la fonction au fichier déjà lu
print(prepared_example)                         # Affiche le résultat

In [ ]:
def prepare_example(example):  # Transforme une annotation en un exemple simplifié
    if example["was_cancelled"]:  # Vérifie si l'annotation a été annulée
        return None               # Indique qu'on ne garde pas cet exemple

    review = get_choices(example, "review_decision")  # Récupère la décision de relecture

    if review != ["Accepter"]:  # Écarte les annotations non acceptées
        return None

    data = example["task"]["data"]  # Récupère les données du message
    intents = get_choices(example, "intents")  # Conserve toutes les intentions
    intent_types = get_choices(example, "type_intent")  # Récupère les types sélectionnés

    if not intents:  # Vérifie qu'au moins une intention est renseignée
        raise ValueError("Cet exemple accepté n'a aucune intention.")

    if len(intent_types) != 1:  # Vérifie qu'il y a exactement un type d'intention
        raise ValueError(f"Un seul type attendu, trouvé : {intent_types}")

    assimilated = get_choices(example, "Assimilated_by_Mode")  # Garde cette information pour le filtrage

    return {  # Construit le dictionnaire représentant notre exemple
        "text": data["message"],              # Texte donné au modèle
        "intents": intents,                   # Liste des intentions à prédire
        "type_intent": intent_types[0],       # Unique type à prédire
        "split": data["split"],               # Groupe d'origine : train ou test
        "Assimilated_by_Mode": assimilated[0] if assimilated else None,  # None si absent
    }


prepared_example = prepare_example(raw_example)  # Applique la fonction au fichier déjà lu
print(prepared_example)                         # Affiche le résultat